# PDF to Chroma Pipeline

### This notebook loads a PDF, splits it into chunks, stores the chunks in Chroma and uses a retrieval query

In [1]:
import os
from pathlib import Path

from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

C:\Users\Athahar\AppData\Local\Temp\ipykernel_10672\2634496628.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [2]:
notebook_dir = Path.cwd()      #current notebook dir
vector_store_dir = notebook_dir.parent    #04_vectors_stores dir
project_root = vector_store_dir.parent    #RAG_Main_Codes dir

In [3]:
env_path = project_root / ".env"
pdf_path = vector_store_dir / "documents" / "beyond-chatbots-ai-agents-next-real-shift.pdf"
persist_directory = vector_store_dir / "notebooks" / "chroma_langchain_db"
collection_name = "rag-pipeline"

print(f"Project root: {project_root}")
print(f"PDF path: {pdf_path}")
print(f"Collection name: {collection_name}")
print(f"Persist directory: {persist_directory}")

Project root: c:\Users\Athahar\Advance_RAG_ath\RAG_Main_codes
PDF path: c:\Users\Athahar\Advance_RAG_ath\RAG_Main_codes\04_vector_stores\documents\beyond-chatbots-ai-agents-next-real-shift.pdf
Collection name: rag-pipeline
Persist directory: c:\Users\Athahar\Advance_RAG_ath\RAG_Main_codes\04_vector_stores\notebooks\chroma_langchain_db


In [4]:
load_dotenv(dotenv_path=env_path)

if not os.getenv("EMB_API_KEY"):
    raise ValueError("Please add your OPENAI_API_KEY to the .env file before running this notebook.")

print(f"Loaded environment from: {env_path}")

Loaded environment from: c:\Users\Athahar\Advance_RAG_ath\RAG_Main_codes\.env


In [5]:
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    api_key=os.environ["EMB_API_KEY"],
    base_url="https://api.euron.one/api/v1/euri",
)

print("Embedding model is ready.")

Embedding model is ready.


# Small display helpers

In [6]:
def preview_text(text, limit=120):
    """Return a short preview for cleaner notebook output."""
    if len(text) <= limit:
        return text
    return text[:limit] + "..."


def print_documents(title, docs):
    """Print retrieved documents using page metadata and a text preview."""
    print(title)
    for index, doc in enumerate(docs, start=1):
        print(f"{index}. page={doc.metadata.get('page')} | source={doc.metadata.get('source')}")
        print(f"   content={doc.page_content}")
    print()

# Load the pdf

In [7]:
loader = PyPDFLoader(str(pdf_path))

In [8]:
docs = loader.load()
print(f"Total pages loaded: {len(docs)}")

Total pages loaded: 6


In [9]:
docs[0].page_content

'Page 1\n Beyond Chatbots: Why AI Agents Feel Like the\n Next Real Shift\nA practical long-form blog on planning, memory, tools, and retrieval in modern AI systems\nBy Editorial Desk\nThe moment AI stopped feeling like a demo\nFor a long time, the most common experience with AI felt theatrical. You typed a question, the model\nanswered in polished language, and for a moment it seemed almost magical. Then the illusion broke. Ask a\nfollow-up that required memory, factual grounding, or a small sequence of actions, and the system often fell\napart. It could sound confident without being connected to anything real. That gap between fluency and\nusefulness is exactly where AI agents enter the picture.\nAn AI agent is interesting not because it sounds human, but because it behaves like software with intent. It\ncan take a goal, figure out what it needs in order to make progress, and work through a sequence of steps\ninstead of improvising a single reply. In the simplest form, that might mean

In [10]:
print(f"First page preview: {preview_text(docs[0].page_content)}")
print(f"First page metadata: {docs[0].metadata}")

First page preview: Page 1
 Beyond Chatbots: Why AI Agents Feel Like the
 Next Real Shift
A practical long-form blog on planning, memory, to...
First page metadata: {'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-03-12T20:36:07+05:00', 'author': 'By Editorial Desk', 'keywords': '', 'moddate': '2026-03-12T20:36:07+05:00', 'subject': '(unspecified)', 'title': 'Beyond Chatbots: Why AI Agents Feel Like the Next Real Shift', 'trapped': '/False', 'source': 'c:\\Users\\Athahar\\Advance_RAG_ath\\RAG_Main_codes\\04_vector_stores\\documents\\beyond-chatbots-ai-agents-next-real-shift.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1'}


# Spli the pdf into chunks

In [11]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
)

In [12]:
chunked_docs = text_splitter.split_documents(docs)
print(f"Total chunks created: {len(chunked_docs)}")

Total chunks created: 88


In [13]:
print(f"First chunk preview: {preview_text(chunked_docs[0].page_content)}")
print(f"First chunk metadata: {chunked_docs[0].metadata}")

First chunk preview: Page 1
 Beyond Chatbots: Why AI Agents Feel Like the
 Next Real Shift
A practical long-form blog on planning, memory, to...
First chunk metadata: {'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-03-12T20:36:07+05:00', 'author': 'By Editorial Desk', 'keywords': '', 'moddate': '2026-03-12T20:36:07+05:00', 'subject': '(unspecified)', 'title': 'Beyond Chatbots: Why AI Agents Feel Like the Next Real Shift', 'trapped': '/False', 'source': 'c:\\Users\\Athahar\\Advance_RAG_ath\\RAG_Main_codes\\04_vector_stores\\documents\\beyond-chatbots-ai-agents-next-real-shift.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1'}


# Store chunks into chroma

In [14]:
collection_name

'rag-pipeline'

In [15]:
persist_directory

WindowsPath('c:/Users/Athahar/Advance_RAG_ath/RAG_Main_codes/04_vector_stores/notebooks/chroma_langchain_db')

In [16]:
vector_store = Chroma.from_documents(
    documents=chunked_docs,
    embedding=embeddings,
    collection_name=collection_name,
    persist_directory=str(persist_directory),
)

print(f"Stored {len(chunked_docs)} chunks in the '{collection_name}' collection.")

Stored 88 chunks in the 'rag-pipeline' collection.


# Retrieve Relevant Chunks

In [17]:
query = "How do AI agents use tools and memory?"
query

'How do AI agents use tools and memory?'

In [18]:
results = vector_store.similarity_search(query, k=3)

print(f"Query: {query}\n")
print_documents("Retrieved chunks:", results)

Query: How do AI agents use tools and memory?

Retrieved chunks:
1. page=2 | source=c:\Users\Athahar\Advance_RAG_ath\RAG_Main_codes\04_vector_stores\documents\beyond-chatbots-ai-agents-next-real-shift.pdf
   content=reveals the reasoning surface of the system. The answer no longer feels like a black box. It feels like the
product of a process that can be inspected.
Why tools make agents actually useful
If memory gives an agent context, tools give it reach. A model can describe a search, but a tool can perform
2. page=2 | source=c:\Users\Athahar\Advance_RAG_ath\RAG_Main_codes\04_vector_stores\documents\beyond-chatbots-ai-agents-next-real-shift.pdf
   content=Page 3
Memory is not just a feature, it is a design decision
When people first hear that agents need memory, they sometimes imagine a giant transcript that the model
carries forever. In practice, memory is much more selective than that. Good systems decide what should stay
3. page=0 | source=c:\Users\Athahar\Advance_RAG_ath\RAG_Main

In [19]:
retrieved_docs = vector_store.similarity_search_with_score(query, k=2)

for doc, score in retrieved_docs:
    print(f"Score: {score:.4f}")
    print(f"Content preview: {doc.page_content}")
    print(f"page_no. {doc.metadata.get("page_label")}")
    print()

Score: 0.8495
Content preview: reveals the reasoning surface of the system. The answer no longer feels like a black box. It feels like the
product of a process that can be inspected.
Why tools make agents actually useful
If memory gives an agent context, tools give it reach. A model can describe a search, but a tool can perform
page_no. 3

Score: 0.8948
Content preview: Page 3
Memory is not just a feature, it is a design decision
When people first hear that agents need memory, they sometimes imagine a giant transcript that the model
carries forever. In practice, memory is much more selective than that. Good systems decide what should stay
page_no. 3

